# Extract data from CERCA raw files

In [2]:
from docx import Document
import pandas as pd
from google.cloud import bigquery
import requests
from tqdm import tqdm
import time
from df2gspread import gspread2df as g2d

In [3]:
interest_centers = ['BETA']
file_path = '../data/external/5_Bibliometria_SIRIS_081025/'

## Extract DOI list

**Each file has a different format so we go center by center**

### BETA

In [4]:
center_name = 'BETA/'
file_name = 'LIST OF SCIENTIFIC PUBLICATIONS_BETA_Updated'

In [5]:
doc = Document(file_path + 'BM_' + center_name + file_name + '.docx')
pubs = [para.text for para in doc.paragraphs]
DOI_tmp = [pub.split('DOI:', 1)[1].strip() for pub in pubs  if 'DOI:' in pub]
DOI_BETA = [(DOI.split('https://doi.org/', 1)[1].strip() if 'https://doi.org/' in DOI else DOI) for DOI in DOI_tmp]
df_BETA = pd.DataFrame(DOI_BETA, columns = ['DOI'])
df_BETA['Center'] = 'BETA'
df_BETA

,DOI,Center
0,10.1016/j.scitotenv.2023.168824,BETA
1,10.23818/limn.43.07,BETA
2,10.1016/j.aquatox.2024.106843,BETA
3,10.3390/agronomy14050935,BETA
4,10.32347/2077-3455.2024.68.215-227,BETA
...,...,...
144,10.1007/s00265-021-03002-7,BETA
145,10.1007/s10841-021-00307-w,BETA
146,10.1016/j.ecoenv.2020.111215,BETA
147,10.1073/pnas.210276211,BETA


In [6]:
df_centers = df_BETA
df_centers

,DOI,Center
0,10.1016/j.scitotenv.2023.168824,BETA
1,10.23818/limn.43.07,BETA
2,10.1016/j.aquatox.2024.106843,BETA
3,10.3390/agronomy14050935,BETA
4,10.32347/2077-3455.2024.68.215-227,BETA
...,...,...
144,10.1007/s00265-021-03002-7,BETA
145,10.1007/s10841-021-00307-w,BETA
146,10.1016/j.ecoenv.2020.111215,BETA
147,10.1073/pnas.210276211,BETA


## Check which publications are not in OA using DOI

In [7]:
PROJECT_ID = 'siris-datasets'
DATASET_ID = 'openalex'

def bg_query(query):
    client = bigquery.Client(project=PROJECT_ID)
    df = client.query(query)
    return df.to_dataframe()

In [8]:
in_query = str(tuple(df_centers.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       wa.author_order,
       wa.author_position,
       wa.is_corresponding,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      WHERE ww.DOI IN {in_query}
      """
df_OA = bg_query(sql).dropna(subset = 'DOI').reset_index(drop = True)
df_OA

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
0,10.2993/0278-0771-41.1.53,Eric Marcel Temba,6,middle,False,137724175,MG
1,10.2993/0278-0771-41.1.53,Eric Marcel Temba,6,middle,False,133731052,FI
2,10.1111/1365-2656.13689,Pau Colom,2,middle,False,4210116941,ES
3,10.1038/s41597-024-03611-7,Amanda H. Korstjens,129,middle,False,9300472,GB
4,10.1038/s41597-024-03611-7,Christopher J. Watson,253,middle,False,63341726,CA
...,...,...,...,...,...,...,...
2114,10.3390/d13090454,Andreu Ubach,15,middle,False,4210111059,ES
2115,10.3390/membranes12090848,Mabel Mora,8,middle,False,115304662,ES
2116,10.1016/j.jnc.2022.126177,Diogo F. Ferreira,1,first,True,182534213,PT
2117,10.1016/j.jnc.2022.126177,Diogo F. Ferreira,1,first,True,45129253,GB


In [9]:
df_OA.DOI.nunique()

136

### Identify CERCA authors using OA (93% of the dataset)

- By affiliation ID
- By raw affiliations
  - Using parents affiliations
  - Manually checking above > 1 per raw affiliation [2k affiliations]
  - String search with keywords for = 1 doi per raw affilation [4k affilations]

**We identify 70% of the provided DOI's**

For the ResearchMar the retrieval is 61%; being the center with most publications we will force the manual identification to increase it by selecting in a second phase using a manual validation [word mar and Barcelona]. We reach 67%

The problem is the hypothesis of using the parent affiliations (problem for the hospital) that is not good enough but I can't manually review all the affiliations of spain. So I can't improve it further

So to solve it we select the not found dois in Researchmar using the previous parent affiliations hypothesis and chec the raw affiliations in Spain looking for strings (mar, imim, parc combined with barcelona without checking).We reach 84%

In [10]:
cerca_centers = {'BETA' : [''], # NOT IN OA
                    'CREAF' : ['4210129656', '4401200259'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}

interest_cerca = list(set(sum(list(cerca_centers.values()), [])))
interest_cerca = [int(x) for x in interest_cerca if x != '']

cerca_authors = df_OA[df_OA.institution_id.isin(interest_cerca)].drop_duplicates(['DOI'])

cerca_parents = {'BETA' : ['115304662'],
                    'CREAF' : ['123044942', '71999127'],
                    'ICN2' : ['123044942'],
                    'ISGlobal' : ['123044942', '170486558'],
                    'ResearchMar' : ['170486558']}

parents_cerca = list(set(sum(list(cerca_parents.values()), [])))
parents_cerca = [int(x) for x in parents_cerca if x != '']

cerca_possible_authors = df_OA[(~df_OA.display_name.isin(cerca_authors.display_name))].drop_duplicates(['DOI'])
cerca_possible_authors

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
0,10.2993/0278-0771-41.1.53,Eric Marcel Temba,6,middle,False,137724175,MG
2,10.1111/1365-2656.13689,Pau Colom,2,middle,False,4210116941,ES
3,10.1038/s41597-024-03611-7,Amanda H. Korstjens,129,middle,False,9300472,GB
6,10.1111/acv.12636,Christian C. Voigt,24,middle,False,4210154298,DE
7,10.23818/limn.43.07,Ignacio Pérez‐Silos,34,middle,False,<NA>,None
...,...,...,...,...,...,...,...
1108,10.1515/mammalia-2020-0056,Ignasi Torre,3,last,True,4210111059,ES
1109,10.1007/s13364-023-00720-3,Ignasi Torre,2,last,False,4210111059,ES
1156,10.1007/s10344-024-01828-w,Mariano J. Feldman,3,middle,False,<NA>,None
1179,10.1007/978-3-031-43071-8_14,Christoph F. J. Meyer,4,last,False,187079419,BR


In [12]:
in_query = str(tuple(cerca_possible_authors.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       war.raw_affiliation,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.DOI IN {in_query}
      """
df_possible_cerca = bg_query(sql).dropna(subset = 'DOI').drop_duplicates(['DOI', 'raw_affiliation']).reset_index(drop = True).reset_index(drop = True)
df_possible_cerca

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,raw_affiliation,institution_id,COUNTRY_CODE
0,10.3390/d13040155,Fèlix Amat,"Area d’Herpetologia, BiBIO, Museu de Granoller...",4210100170,ES
1,10.1016/j.envpol.2022.120127,Carmen Espinosa,"BETA Technological Center, University of Vic- ...",115304662,ES
2,10.1038/s41597-024-03611-7,Georges Reckinger,"Schroeder & Associés, Kockelscheuer, Luxembourg",<NA>,None
3,10.1080/14735903.2024.2361578,Laia Llenas,"BETA Technological Center, TECNIO Network, Uni...",115304662,ES
4,10.1111/mam.12369,Giorgio Zavattoni,"Department of Biology, University of Turku, 20...",155660961,FI
...,...,...,...,...,...
1064,10.1002/ecy.3614,Alfonso Allen‐Perkins,"Departamento de Ingeniería Eléctrica, Electrón...",88060688,ES
1065,10.1016/j.sajb.2021.06.035,Gonzalo Sacristán,"Laboratory of Microbiology, Faculty of Science...",46176106,ES
1066,10.1007/s00265-021-03002-7,Joan Martí‐Carreras,"Department of Microbiology, Immunology and Tra...",99464096,BE
1067,10.1111/acv.12636,Raina K. Plowright,"Department of Microbiology and Immunology, Mon...",23732399,US


In [14]:
grouped = df_possible_cerca[df_possible_cerca.COUNTRY_CODE == 'ES'].groupby('raw_affiliation').count().sort_values('DOI', ascending = False)[['DOI']]
to_check = grouped[grouped.DOI > 1]
print(df_possible_cerca[df_possible_cerca.raw_affiliation.isin(to_check.index)].DOI.nunique())
to_check.to_csv('to_check_BETA.csv')
to_check

33


,DOI
raw_affiliation,
"BiBio Research Group, Natural Sciences Museum of Granollers, C/Francesc Macià 51, E-08402 Granollers, Spain",3
"Natural Sciences Museum of Granollers, Granollers, Catalonia, Spain",3
"Natural Sciences Museum of Granollers, Granollers, Spain",3
"BiBio Research Group, Natural Sciences Museum of Granollers, Granollers, Spain",2
"BiBio Research Group, Natural Sciences Museum of Granollers, Francesc Macià 51, 08402, Granollers, Spain",2
"CERM, Center for the Study of Mediterranean Rivers, University of Vic – Central University of Catalonia (UVic-UCC), Manlleu, Spain",2
Museu de Ciències Naturals de Granollers Granollers Spain,2
"IMDEA Water Institute, Avenida Punto Com, 2, Alcalá de Henares, 28805 Madrid, Spain",2
"BETA Technological Center, University of Vic–University of Central Catalonia, Carrer de la Laura 13, 08500, Vic, Catalonia, Spain",2


In [15]:
df_check = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', '>1BETA', col_names = True, row_names = False)

compute  = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]

len(set(list(compute.DOI.unique()) + list(cerca_authors.DOI.unique()))) / df_centers.DOI.nunique()

Not all requested scopes were granted by the authorization server, missing scopes https://docs.google.com/feeds, https://spreadsheets.google.com/feeds.


0.2671232876712329

In [16]:
cerca_string = {'BETA' : ['beta']}
cerca_string = list(set(sum(list(cerca_string.values()), [])))


df_check_2 = grouped[(grouped.DOI == 1) & (~grouped.index.isin(df_check.raw_affiliation))].reset_index()
df_check_2['raw_affiliation'] = df_check_2['raw_affiliation'].str.lower()

df_check_2.to_csv('to_check_BETA_1.csv')
df_check_2

# df_check_2['CERCA'] = df_check_2['raw_affiliation'].str.contains(('|'.join(cerca_string)), case=False, na=False).map({True: 'TRUE', False: 'FALSE'})
# df_check_2

,raw_affiliation,DOI
0,"global change and conservation (gcc), organism...",1
1,"granollers natural sciences museum, granollers...",1
2,department of biogeography and global change (...,1
3,instituto cavanilles de biodiversidad y biolog...,1
4,"department of conservation biology, doñana bio...",1
...,...,...
253,"creaf, barcelona, spain",1
254,creaf universitat autònoma de barcelona cerdan...,1
255,creaf cerdanyola del vallés spain,1
256,creaf (centre for ecological research and fore...,1


In [17]:
df_check_2 = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', '=1BETA', col_names = True, row_names = False)

compute  = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_2[df_check_2.CERCA == 'TRUE'].raw_affiliation)]

len(set(list(compute.DOI.unique()) + list(cerca_authors.DOI.unique()))) / df_centers.DOI.nunique()

Not all requested scopes were granted by the authorization server, missing scopes https://docs.google.com/feeds, https://spreadsheets.google.com/feeds.


0.678082191780822

In [18]:
check_1 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]
check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_2[df_check_2.CERCA == 'TRUE'].raw_affiliation)]
# check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_2[df_check_2.CERCA == 'TRUE'].raw_affiliation)]

cerca_doi = list(set(
    list(cerca_authors.DOI.unique()) +
    # list(cerca_possible_authors.DOI.unique()) +
    list(check_1.DOI.unique()) +
    list(check_2.DOI.unique())
))
len(cerca_doi) / df_centers.DOI.nunique()

0.821917808219178

In [19]:
df_check = pd.concat((cerca_authors, check_1, check_2))
df_check

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,raw_affiliation
17,10.1002/ecm.1561,Javier Sala‐Garcia,2,middle,False,4210129656,ES,NaN
56,10.1021/acsestwater.1c00192,Arben Merkoçi,13,last,True,4210093216,ES,NaN
148,10.1111/1365-2435.14121,José Alberto Ramírez‐Valiente,4,middle,False,4210129656,ES,NaN
206,10.1111/geb.13527,Lluís Brotóns,5,middle,False,4210129656,ES,NaN
367,10.3390/land11030449,Albert Montori,6,last,True,4210129656,ES,NaN
...,...,...,...,...,...,...,...,...
1000,10.1016/j.gecco.2022.e02194,Elena Abella,<NA>,NaN,<NA>,115304662,ES,"Technological Centre in Biodiversity, Ecology ..."
1028,10.3390/fire6010034,Ignasi Torre,<NA>,NaN,<NA>,4210111059,ES,"BiBio Research Group and Mammal Research Area,..."
1030,10.3390/su14031562,Ignasi Torre,<NA>,NaN,<NA>,4210111059,ES,"BiBio Research Group, Natural Sciences Museum ..."
1045,10.1002/rra.4211,David López‐Bosch,<NA>,NaN,<NA>,4210111059,ES,BiBio Research Group Natural Sciences Museum o...


In [20]:
# HI HA ERROR AMB L'ASSIGNACIÓ I PER AIXÒ SURT RAR! S'HA DE FER ALS POSSIBLE (PATENT I NO PARENT) I LLAVORS FER EL MERGE AMB EL OA QUE TÉ L'AUTHOR ORDER

df_OA['CERCA'] = (df_OA['display_name'].isin(df_check['display_name']) & df_OA['DOI'].isin(df_check['DOI']))
df_final = df_OA.merge(df_centers, on = 'DOI').drop_duplicates().reset_index(drop = True)
df_final.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_BETA.csv', index = False)
df_final

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.2993/0278-0771-41.1.53,Eric Marcel Temba,6,middle,False,137724175,MG,False,BETA
1,10.2993/0278-0771-41.1.53,Eric Marcel Temba,6,middle,False,133731052,FI,False,BETA
2,10.1111/1365-2656.13689,Pau Colom,2,middle,False,4210116941,ES,False,BETA
3,10.1038/s41597-024-03611-7,Amanda H. Korstjens,129,middle,False,9300472,GB,False,BETA
4,10.1038/s41597-024-03611-7,Christopher J. Watson,253,middle,False,63341726,CA,False,BETA
...,...,...,...,...,...,...,...,...,...
2114,10.3390/d13090454,Andreu Ubach,15,middle,False,4210111059,ES,True,BETA
2115,10.3390/membranes12090848,Mabel Mora,8,middle,False,115304662,ES,False,BETA
2116,10.1016/j.jnc.2022.126177,Diogo F. Ferreira,1,first,True,182534213,PT,False,BETA
2117,10.1016/j.jnc.2022.126177,Diogo F. Ferreira,1,first,True,45129253,GB,False,BETA
